In [1]:
import pandas as pd

In [ ]:
# load excerpt with only two vehicles to df

file_path = 'two_cars.csv'

# Create final DataFrame
columns = ['Track ID', 'Type', 'x [m]', 'y [m]', 'Speed [km/h]',
        'Tan. Acc. [ms-2]', 'Lat. Acc. [ms-2]', 'Time [s]', 'Angle [rad]']

df = pd.read_csv(file_path, delimiter=",")

df.head()


,Track ID,Type,x [m],y [m],Speed [km/h],Tan. Acc. [ms-2],Lat. Acc. [ms-2],Time [s],Angle [rad]
0,1,Car,411374.81,5655227.47,57.8746,-2.2080,-0.3367,0.066733,0.8582
1,1,Car,411375.17,5655227.89,57.6237,-1.9688,-0.3644,0.100100,0.8584
2,1,Car,411375.53,5655228.31,57.4030,-1.7065,-0.4042,0.133467,0.8586
3,1,Car,411375.89,5655228.73,57.2113,-1.4853,-0.4354,0.166833,0.8589
4,1,Car,411376.25,5655229.14,57.0482,-1.2287,-0.4692,0.200200,0.8592


## Conversion

The positional data in `two_cars.csv` is in the [UTM format](https://en.wikipedia.org/wiki/Universal_Transverse_Mercator_coordinate_system), specifically it refers to UTM zone 33, where `x`= Easting and `y`= Northing.
We need to convert it to x-y-positions in meter that are relative to the origin of the `xodr` map that we want to use, `map_hauptbahnhof_nord_dresden.osm`.

To do that, we use the `pyproj` package.


In [3]:
from pyproj import CRS, Transformer
from typing import Tuple


def convert_utm_to_local(
        easting: float, 
        northing: float,
        reference_lat: float=51.040659, 
        reference_lon: float=13.733844,
        utm_zone: int=33) -> Tuple[float, float]:
    """
    Converts UTM trajectory to local coordinates aligned with osm2xodr's tmerc projection.
    
    Parameters:
        df: pandas DataFrame with UTM columns
        utm_x_col: column name for UTM X (easting)
        utm_y_col: column name for UTM Y (northing)
        reference_lat: lat_0 from your .xodr geoReference (SW corner of OSM)
        reference_lon: lon_0 from your .xodr geoReference
        utm_zone: UTM zone of your trajectory (default: 33 for Dresden area)

    Returns:
        x_local, y_local
    """
    
    utm_crs = CRS.from_epsg(32600 + utm_zone) 
    tmerc_proj = f"+proj=tmerc +lat_0={reference_lat} +lon_0={reference_lon} +x_0=0 +y_0=0 +ellps=GRS80 +units=m +no_defs"
    local_crs = CRS.from_proj4(tmerc_proj)
    utm_to_local = Transformer.from_crs(utm_crs, local_crs, always_xy=True)

    x_local, y_local = utm_to_local.transform(easting, northing)

    return x_local, y_local



In [4]:
reference_lat, reference_lon = (51.0374498, 13.730641) # see `readme.md`

df[["x_loc [m]", "y_loc [m]"]] = df.apply(
    lambda row: pd.Series(
        convert_utm_to_local(
            row["x [m]"], 
            row["y [m]"], 
            reference_lat=reference_lat, 
            reference_lon=reference_lon
        )
    ), 
    axis=1
)

df.head()

,Track ID,Type,x [m],y [m],Speed [km/h],Tan. Acc. [ms-2],Lat. Acc. [ms-2],Time [s],Angle [rad],x_loc [m],y_loc [m]
0,1,Car,411374.81,5655227.47,57.8746,-2.2080,-0.3367,0.066733,0.8582,363.918235,477.886076
1,1,Car,411375.17,5655227.89,57.6237,-1.9688,-0.3644,0.100100,0.8584,364.271053,478.312345
2,1,Car,411375.53,5655228.31,57.4030,-1.7065,-0.4042,0.133467,0.8586,364.623871,478.738614
3,1,Car,411375.89,5655228.73,57.2113,-1.4853,-0.4354,0.166833,0.8589,364.976689,479.164883
4,1,Car,411376.25,5655229.14,57.0482,-1.2287,-0.4692,0.200200,0.8592,365.329679,479.581151


## Export the required fields for a single vehicle

In [5]:
# df[df["Track ID"] == 2].to_csv('2_vehicle_local_coords.csv', index=False, columns=["x_loc [m]", "y_loc [m]", "Time [s]"])

# Extract arbitrary trajectories from the full excerpt

In [ ]:
full_excerpt_path = "20220511_100012_Sid_StP_3W_d_1_1_ann.csv"
cols = "Track ID; Type; Track Width [m]; Track Width Displacement [m]; Track Length [m]; Track Length Displacement [m]; Entry Gate; Entry Time [s]; Exit Gate; Exit Time [s]; Traveled Dist. [m]; Avg. Speed [km/h]; Trajectory(x [m]; y [m]; Speed [km/h]; Tan. Acc. [ms-2]; Lat. Acc. [ms-2]; Time [s]; Angle [rad]; )" # this is the first line from the excerpt
cols = [c.strip() for c in cols.replace("(", "").replace(")", "").replace("Trajectory", "").strip().split(";")[:-1]]
trajectory_cols = cols[-7:] # these are the columns that designate the trajectory

In [7]:
trajectories = {}

with open(full_excerpt_path, "r") as f:

    #every line corresponds to one trajectory
    for line_number,line in enumerate(f):
        if line_number == 0:
            continue # skip the header
        print(f"line {line_number}, with {len(line.split(';'))} tokens")
        trajectory_id = line_number
        tokens = line.split(";")
        # we need to separate the trajectory from the trajectory metadata in tokens 0 to 11
        for token_number, token in enumerate(tokens):
            token = token.strip()
            if token_number == 0 or 1 < token_number < 12:
                continue
            elif token_number == 1:
                if token in ["Car", "Bus", "Medium Vehicle"]:
                    trajectories[trajectory_id] = {"type": token, "x": [], "y": [], "t": []}
                else: 
                    break # if trajectory does not belong to wanted class, break
            else: 
                trajectory_part = (token_number - 12) % 7
                try:
                    if trajectory_part == 0: # x [m]
                        trajectories[trajectory_id]["x"].append(float(token))
                    elif trajectory_part == 1: # y [m]
                        trajectories[trajectory_id]["y"].append(float(token))
                    elif trajectory_part == 5: # Time [s]
                        trajectories[trajectory_id]["t"].append(float(token))
                    else:
                        continue
                except Exception as e: 
                    print(f"Exception encountered at token {token_number}, trying to continue: {e}")





line 1, with 531 tokens
line 2, with 6635 tokens
line 3, with 1084 tokens
Exception encountered at token 1083, trying to continue: could not convert string to float: ''
line 4, with 769 tokens
Exception encountered at token 768, trying to continue: could not convert string to float: ''
line 5, with 1273 tokens
Exception encountered at token 1272, trying to continue: could not convert string to float: ''
line 6, with 6754 tokens
Exception encountered at token 6753, trying to continue: could not convert string to float: ''
line 7, with 447 tokens
Exception encountered at token 446, trying to continue: could not convert string to float: ''
line 8, with 15077 tokens
Exception encountered at token 15076, trying to continue: could not convert string to float: ''
line 9, with 67997 tokens
Exception encountered at token 67996, trying to continue: could not convert string to float: ''
line 10, with 2092 tokens
Exception encountered at token 2091, trying to continue: could not convert string to 

In [8]:
# Check whether for every trajectory id, the x, y and t lists have the same length

for trajectory_id in trajectories:
    len_x = len(trajectories[trajectory_id]["x"])
    len_y = len(trajectories[trajectory_id]["y"])
    len_t = len(trajectories[trajectory_id]["t"])
    if len_x != len_y or len_x != len_t or len_y != len_t:
        print(f"Mismatch at {trajectory_id}")

## Convert trajectory coordinates from UTM to local

In [9]:
# reference_lat, reference_lon = (51.0374498, 13.730641) # see `readme.md`

# for trajectory_id in trajectories:
#     xs = trajectories[trajectory_id]["x"]
#     ys = trajectories[trajectory_id]["y"]
#     ts = trajectories[trajectory_id]["t"]

#     trajectories[trajectory_id]["local_coords"] = []

#     print(f"Processing trajectory {trajectory_id}")

#     for i in range(len(xs)):
#         x_local, y_local = convert_utm_to_local(xs[i], ys[i], reference_lat, reference_lon)
#         trajectories[trajectory_id]["local_coords"].append((x_local, y_local, ts[i]))

In [10]:
# make a fat parallelized conversion to local coordinates

from concurrent.futures import ProcessPoolExecutor, as_completed

reference_lat, reference_lon = (51.0374498, 13.730641)  # see `readme.md`

def process_trajectory(traj_id_and_data):
    trajectory_id, data = traj_id_and_data
    xs, ys, ts = data["x"], data["y"], data["t"]
    
    local_coords = [
        (*convert_utm_to_local(xs[i], ys[i], reference_lat, reference_lon), ts[i])
        for i in range(len(xs))
    ]
    
    return trajectory_id, local_coords


with ProcessPoolExecutor() as executor:
    futures = [executor.submit(process_trajectory, (tid, trajectories[tid])) for tid in trajectories]

    completed_count = 0
    for future in as_completed(futures):
        trajectory_id, local_coords = future.result()
        completed_count += 1
        print(f"Processed {completed_count} trajectories")
        trajectories[trajectory_id]["local_coords"] = local_coords

Processed 1 trajectories
Processed 2 trajectories
Processed 3 trajectories
Processed 4 trajectories
Processed 5 trajectories
Processed 6 trajectories
Processed 7 trajectories
Processed 8 trajectories
Processed 9 trajectories
Processed 10 trajectories
Processed 11 trajectories
Processed 12 trajectories
Processed 13 trajectories
Processed 14 trajectories
Processed 15 trajectories
Processed 16 trajectories
Processed 17 trajectories
Processed 18 trajectories
Processed 19 trajectories
Processed 20 trajectories
Processed 21 trajectories
Processed 22 trajectories
Processed 23 trajectories
Processed 24 trajectories
Processed 25 trajectories
Processed 26 trajectories
Processed 27 trajectories
Processed 28 trajectories
Processed 29 trajectories
Processed 30 trajectories
Processed 31 trajectories
Processed 32 trajectories
Processed 33 trajectories
Processed 34 trajectories
Processed 35 trajectories
Processed 36 trajectories
Processed 37 trajectories
Processed 38 trajectories
Processed 39 trajecto

In [16]:
# Export

for trajectory_id, trajectory_data in trajectories.items():
    vehicle_type = trajectory_data["type"]
    local_coords = trajectory_data["local_coords"]
    with open(f"{trajectory_id}_{vehicle_type}_local_coords.csv", "w") as openf:
        print("x_loc [m],y_loc [m],Time [s]", file=openf)

        for x,y,t in local_coords:
            print(x, y, t, sep=",", file=openf)